# Data Preparation — Training Manifest with Named Entities

In [ ]:
!pip install -U spacy transformers torch torchvision jiwer pandas tqdm torchcodec
!python -m spacy download en_core_web_trf   # or de_dep_news_trf fr_dep_news_trf es_dep_news_trf


## Load the Kaldi Dataset

In [ ]:
import torchaudio
from IPython.display import Audio, display


dataset = []

dataset_path = "/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test" #other languages *_de *_fr *_es
# other datasets
# Mozilla Commonvoice: /training-1/asr/kaldi_data/cv19/en/test /de/ /fr/ /es/
# Mozilla Commonvoice Single Words: /training-1/asr/kaldi_data/singleword/en/test /de/ /fr/ /es/ 
# Multilingual Librispeech: /training-1/asr/kaldi_data/mls/en/test /de/ /fr/ /es/
# Voxpopuli: /training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test *_de *_fr *_es

wav_scp_dict = {}
with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
    for line in rf:
        id, wav_path = line.strip().split("\t")
        wav_scp_dict[id] = wav_path

text_dict = {}
with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
    for line in rf:
        id, text = line.strip().split("\t")
        text_dict[id] = text

for id, wav_path in wav_scp_dict.items():
    text = text_dict[id]
    dataset.append({"audio" : wav_path, "text" : text})

#helper function: load audio and make sure it is at 16kHz
def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform
    
first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = load_audio(first_entry["audio"])
display(Audio(audio, rate=16000))


## Named Entity Extraction

In [ ]:
import spacy

nlp = spacy.load("en_core_web_trf")   # or de_dep_news_trf fr_dep_news_trf es_dep_news_trf

def extract_entities(text):
    return [(e.text, e.label_) for e in nlp(text).ents]

def build_prompt(text):
    """Build a Qwen3-ASR biasing prompt from the entities in the text."""
    seen = []
    for e, _ in extract_entities(text):
        if e not in seen:
            seen.append(e)
    if not seen:
        return None
    return "Vocabulary: " + ", ".join(seen) + "."

print(extract_entities(first_entry["text"]))
print(build_prompt(first_entry["text"]))

## Train/Dev Split

In [ ]:
import random

random.seed(42)

shuffled = dataset.copy()
random.shuffle(shuffled)

dev_ratio = 0.1   # 10 % of the samples for the dev split
n_dev = max(1, int(len(shuffled) * dev_ratio))

dev_data = shuffled[:n_dev]
train_data = shuffled[n_dev:]

print(f"{len(train_data)} train samples, {len(dev_data)} dev samples")

## Write the Training and Dev Manifests

In [ ]:
import json

MANIFEST_DIR = Path("/training-1/asr_finetune")   # <-- adapt
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_TRAIN = MANIFEST_DIR / "manifest_train.jsonl"
MANIFEST_DEV = MANIFEST_DIR / "manifest_dev.jsonl"

def write_manifest(samples, path):
    """Write one JSON line per sample: audio path, transcript and entity prompt."""
    with open(path, "w", encoding="UTF-8") as wf:
        for sample in samples:
            entry = {"audio": sample["audio"], "text": sample["text"],
                     "prompt": build_prompt(sample["text"])}
            wf.write(json.dumps(entry, ensure_ascii=False) + "\n")

write_manifest(train_data, MANIFEST_TRAIN)
write_manifest(dev_data, MANIFEST_DEV)

print(f"Wrote {len(train_data)} entries to {MANIFEST_TRAIN}")
print(f"Wrote {len(dev_data)} entries to {MANIFEST_DEV}")

## Inspect the Manifests

In [ ]:
for path in [MANIFEST_TRAIN, MANIFEST_DEV]:
    print("=" * 70)
    print(path)
    with open(path, encoding="UTF-8") as rf:
        for i, line in enumerate(rf):
            if i >= 5:
                break
            print(json.loads(line))